<a href="https://colab.research.google.com/github/lmVl12/AI_and_Drug_Discovery_Course_2026/blob/main/Assignment%203/Assignment_3_Task2_2D_descriptors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **AI And Biotechnology/Bioinformatics**

## **AI and Drug Discovery Course: QSAR Modeling**
This notebook demonstrates how to collect and preprocess bioactivity data from ChEMBL for QSAR modeling

# **Part 3: Descriptor Calculation**

The selection of a descriptor calculation tool depends on the requirement for a high-dimensional feature space and the robustness of the software environment. In this study, a 'process of elimination' was applied to choose the most suitable tool:

* Commercial software (e.g., **Dragon, alvaDesc**), while offering the most extensive descriptor sets (5,000+), was excluded due to licensing constraints.

* Basic libraries (e.g., **RDKit**) provide high-quality data but are limited in the variety of topological indices (~200 descriptors), which may not capture sufficient structural complexity for this target.

* **Mordred** emerged as a strong candidate with over 1,800 descriptors; however, it was dismissed due to significant technical limitations, specifically its dependency on outdated software versions and lack of active maintenance, which poses risks to reproducibility in modern environments.

Consequently, **PaDEL-Descriptor** was selected as the optimal solution. It offers a balanced set of 1,444 descriptors and, crucially, operates as a standalone tool. Its autonomy from the versioning of underlying cheminformatics toolkits ensures greater stability and reliability for the QSAR modeling pipeline

## **1. Technical Framework**

In [1]:
!pip install padelpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 72.9 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
from google.colab import files
from padelpy import padeldescriptor

## **2. Load dataset**

In [3]:
from google.colab import drive
drive.mount('/content/gdrive/')

Mounted at /content/gdrive/


Loading the preprocessed Lipinski-filtered dataset from Google Drive and inspecting its structure.

In [6]:
results_path = "/content/"
df = pd.read_csv(results_path + 'df_lipinski.csv')
df.head()

,molecule_chembl_id,bioactivity_class,pIC50,canonical_smiles,MW,LogP,NumHDonors,NumHAcceptors
0,CHEMBL392346,inactive,3.000000,CCCCC(=O)O[C@H]1[C@H](O)[C@@H](CO)O[C@H]1n1cc(...,342.348,-0.80238,3.0,7.0
1,CHEMBL401029,inactive,3.301030,CCCCC(=O)O[C@H]1[C@H](O)[C@@H](CO)O[C@H]1n1cnc...,367.362,-0.94560,4.0,9.0
2,CHEMBL134449,inactive,4.958607,CCCCCCC(=O)O/C=C\Cn1cc(C)c(=O)[nH]c1=O,294.351,1.87232,1.0,4.0
3,CHEMBL241166,inactive,3.301030,CCCCCCCC(=O)Nc1nc2c(ncn2[C@@H]2O[C@H](CO)[C@@H...,535.642,2.94150,4.0,9.0
4,CHEMBL238635,inactive,3.000000,CCCCCCCC(=O)O[C@H]1[C@H](O)[C@@H](CO)O[C@H]1n1...,475.336,1.42510,3.0,7.0


In [7]:
data = df[['canonical_smiles', 'molecule_chembl_id']]
data.head()

,canonical_smiles,molecule_chembl_id
0,CCCCC(=O)O[C@H]1[C@H](O)[C@@H](CO)O[C@H]1n1cc(...,CHEMBL392346
1,CCCCC(=O)O[C@H]1[C@H](O)[C@@H](CO)O[C@H]1n1cnc...,CHEMBL401029
2,CCCCCCC(=O)O/C=C\Cn1cc(C)c(=O)[nH]c1=O,CHEMBL134449
3,CCCCCCCC(=O)Nc1nc2c(ncn2[C@@H]2O[C@H](CO)[C@@H...,CHEMBL241166
4,CCCCCCCC(=O)O[C@H]1[C@H](O)[C@@H](CO)O[C@H]1n1...,CHEMBL238635


## **3. Convert to .smi format**

SMILES strings are converted into a .smi format to ensure a standardized input for the PaDEL software, allowing for consistent parsing of chemical structures.

In [8]:
df_smi = data['canonical_smiles'].to_csv('smiles_chembl.smi', index=None, header=None)

In [9]:
! cat smiles_chembl.smi | head

CCCCC(=O)O[C@H]1[C@H](O)[C@@H](CO)O[C@H]1n1cc(C)c(=O)[nH]c1=O
CCCCC(=O)O[C@H]1[C@H](O)[C@@H](CO)O[C@H]1n1cnc2c(=O)[nH]c(N)nc21
CCCCCCC(=O)O/C=C\Cn1cc(C)c(=O)[nH]c1=O
CCCCCCCC(=O)Nc1nc2c(ncn2[C@@H]2O[C@H](CO)[C@@H](O)[C@@H]2OC(=O)CCCCCCC)c(=O)[nH]1
CCCCCCCC(=O)O[C@H]1[C@H](O)[C@@H](CO)O[C@H]1n1cc(/C=C/Br)c(=O)[nH]c1=O
CCCCCCCC(=O)O[C@H]1[C@H](O)[C@@H](CO)O[C@H]1n1cc(C)c(=O)[nH]c1=O
CCCCCCCC(=O)O[C@H]1[C@H](O)[C@@H](CO)O[C@H]1n1cnc2c(=O)[nH]c(N)nc21
CCCCCCCCC(=O)O[C@H]1[C@H](O)[C@@H](CO)O[C@H]1n1ccc(N)nc1=O
CCCCCCCCCC(=O)O[C@H]1[C@H](O)[C@@H](CO)O[C@H]1n1cc(/C=C/Br)c(=O)[nH]c1=O
CCCCCCCCCC(=O)O[C@H]1[C@H](O)[C@@H](CO)O[C@H]1n1cc(C)c(=O)[nH]c1=O


## **4. Calculate molecular 2D descriptors using "padeldescriptor" function**


In [10]:
padeldescriptor(mol_dir="smiles_chembl.smi",
                d_file='descriptors_2d.csv',
                d_2d=True,
                fingerprints=False, # explicit set to false to avoid default calculation
                retainorder=True
                )

In [11]:
!ls -lh descriptors_2d.csv

-rw-r--r-- 1 root root 2.9M Feb 21 17:02 descriptors_2d.csv


In [12]:
df_2d = pd.read_csv("descriptors_2d.csv")
df_2d.head()

,Name,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,nAtom,nHeavyAtom,...,AMW,WTPT-1,WTPT-2,WTPT-3,WTPT-4,WTPT-5,WPATH,WPOL,XLogP,Zagreb
0,AUTOGEN_smiles_chembl_1,0,-1.7723,3.141047,67.7807,48.883446,6,6,46,24,...,7.437885,47.532476,1.980520,25.254489,18.810341,6.444148,1322.0,38.0,0.024,120.0
1,AUTOGEN_smiles_chembl_2,0,-3.0346,9.208797,61.9561,50.714653,9,10,47,26,...,7.811685,52.464047,2.017848,31.636856,16.328601,15.308255,1620.0,42.0,-0.635,136.0
2,AUTOGEN_smiles_chembl_3,0,-1.3078,1.710341,63.0447,46.477446,6,6,43,21,...,6.840883,41.076697,1.956033,16.901539,10.542563,6.358976,1210.0,26.0,2.704,94.0
3,AUTOGEN_smiles_chembl_4,0,-5.1374,26.392879,98.9162,84.212513,9,10,79,38,...,6.775957,76.334158,2.008794,34.978111,18.924309,16.053802,5366.0,56.0,4.875,186.0
4,AUTOGEN_smiles_chembl_5,0,-2.4320,5.914624,88.7802,62.307411,6,6,56,29,...,8.466074,57.566182,1.985041,27.841646,18.893493,6.506431,2409.0,44.0,2.937,140.0


## **5. Prepare Dataset for ML**

In [13]:
df.head()

,molecule_chembl_id,bioactivity_class,pIC50,canonical_smiles,MW,LogP,NumHDonors,NumHAcceptors
0,CHEMBL392346,inactive,3.000000,CCCCC(=O)O[C@H]1[C@H](O)[C@@H](CO)O[C@H]1n1cc(...,342.348,-0.80238,3.0,7.0
1,CHEMBL401029,inactive,3.301030,CCCCC(=O)O[C@H]1[C@H](O)[C@@H](CO)O[C@H]1n1cnc...,367.362,-0.94560,4.0,9.0
2,CHEMBL134449,inactive,4.958607,CCCCCCC(=O)O/C=C\Cn1cc(C)c(=O)[nH]c1=O,294.351,1.87232,1.0,4.0
3,CHEMBL241166,inactive,3.301030,CCCCCCCC(=O)Nc1nc2c(ncn2[C@@H]2O[C@H](CO)[C@@H...,535.642,2.94150,4.0,9.0
4,CHEMBL238635,inactive,3.000000,CCCCCCCC(=O)O[C@H]1[C@H](O)[C@@H](CO)O[C@H]1n1...,475.336,1.42510,3.0,7.0


Zero-variance features are removed from the dataset to eliminate non-informative data, thereby improving computational efficiency and reducing potential model noise

Calculated molecular fingerprints are merged with biological activity labels (pIC50) to construct a finalized training dataset for subsequent QSAR modeling.

In [14]:
from sklearn.feature_selection import VarianceThreshold

# Delete the temporary ID column
X = df_2d.drop(df_2d.columns[0], axis=1)

# Remove constant non-informative descriptors
selector = VarianceThreshold(threshold=0)
X_reduced = selector.fit_transform(X)

# Important - save the original names of descriptors
selected_cols = X.columns[selector.get_support()]
df_2d_clean = pd.DataFrame(X_reduced, columns=selected_cols)

print(f"Total number of 2D descriptors: {X.shape[1]}")
print(f"Number of unique descriptors:   {df_2d_clean.shape[1]}")

# 2. Select only columns for ML
meta_cols = df[['molecule_chembl_id', 'bioactivity_class', 'pIC50']]

meta_cols = meta_cols.reset_index(drop=True)
df_2d_clean = df_2d_clean.reset_index(drop=True)

# Get a full dataset
combined_df = pd.concat([meta_cols, df_2d_clean], axis=1)
combined_df.head()

Total number of 2D descriptors: 1444
Number of unique descriptors:   1151


,molecule_chembl_id,bioactivity_class,pIC50,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,...,AMW,WTPT-1,WTPT-2,WTPT-3,WTPT-4,WTPT-5,WPATH,WPOL,XLogP,Zagreb
0,CHEMBL392346,inactive,3.000000,0.0,-1.7723,3.141047,67.7807,48.883446,6.0,6.0,...,7.437885,47.532476,1.980520,25.254489,18.810341,6.444148,1322.0,38.0,0.024,120.0
1,CHEMBL401029,inactive,3.301030,0.0,-3.0346,9.208797,61.9561,50.714653,9.0,10.0,...,7.811685,52.464047,2.017848,31.636856,16.328601,15.308255,1620.0,42.0,-0.635,136.0
2,CHEMBL134449,inactive,4.958607,0.0,-1.3078,1.710341,63.0447,46.477446,6.0,6.0,...,6.840883,41.076697,1.956033,16.901539,10.542563,6.358976,1210.0,26.0,2.704,94.0
3,CHEMBL241166,inactive,3.301030,0.0,-5.1374,26.392879,98.9162,84.212513,9.0,10.0,...,6.775957,76.334158,2.008794,34.978111,18.924309,16.053802,5366.0,56.0,4.875,186.0
4,CHEMBL238635,inactive,3.000000,0.0,-2.4320,5.914624,88.7802,62.307411,6.0,6.0,...,8.466074,57.566182,1.985041,27.841646,18.893493,6.506431,2409.0,44.0,2.937,140.0


## **6. Check for completeness of dataset after calculation**

In [15]:
nan_count = combined_df.isnull().sum().sum()
if nan_count > 0:
    print(f" {nan_count} empty values found and reduced")
    combined_df = combined_df.dropna()
else:
    print("No missing values")

print(f"Final records count: {combined_df.shape[0]}")

No missing values
Final records count: 158


## **7. Save and download the dataset**

In [16]:
# Save as CSV
combined_df.to_csv(results_path + 'QSAR_dataset_2d.csv', index=False)
print("Combined dataset saved as QSAR_dataset_2d.csv")

# Download file in Colab
files.download(results_path +'QSAR_dataset_2d.csv' )

Combined dataset saved as QSAR_dataset_2d.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>